In [1]:
# ============================================
# 04 - PRETRAINED MODELS
# ============================================

# Standard Python libraries
import os
import random
import numpy as np

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

# Torchvision pretrained models
from torchvision import models

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================
# REPRODUCIBILITY
# ============================================

# Fixed seed makes our experiments more reproducible
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================
# DEVICE
# ============================================

# Use GPU if available.
# Otherwise, automatically fall back to CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================
# PROJECT CONFIGURATION
# ============================================

# Number of disease categories
NUM_CLASSES = 6

# Input image size used during data preparation
IMAGE_SIZE = 224

# Number of training epochs.
# We will tune this later.
NUM_EPOCHS = 10


# ============================================
# DISPLAY CONFIGURATION
# ============================================

print("PyTorch version:", torch.__version__)
print("Device:", device)
print("Number of classes:", NUM_CLASSES)
print("Image size:", IMAGE_SIZE)
print("Epochs:", NUM_EPOCHS)

PyTorch version: 2.9.1+cu126
Device: cuda
Number of classes: 6
Image size: 224
Epochs: 10


In [2]:
# ============================================
# CHECK GPU AVAILABILITY
# ============================================

import torch

# Display the installed PyTorch version
print("PyTorch version:", torch.__version__)

# Check whether this PyTorch installation
# has CUDA support
print("CUDA available:", torch.cuda.is_available())

# Display the CUDA version supported by
# the installed PyTorch build, if available
print("PyTorch CUDA version:", torch.version.cuda)

# If CUDA is available, display the GPU name
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    
    # Display the number of GPUs detected
    print("Number of GPUs:", torch.cuda.device_count())
else:
    print("No CUDA GPU is currently available to PyTorch.")

PyTorch version: 2.9.1+cu126
CUDA available: True
PyTorch CUDA version: 12.6
GPU: NVIDIA GeForce GTX 1650
Number of GPUs: 1


In [3]:
# ============================================
# PRETRAINED MODEL CONFIGURATION
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models

# ============================================
# DEVICE
# ============================================

# Automatically use the NVIDIA GPU when CUDA
# is available. Otherwise, fall back to CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================
# PROJECT SETTINGS
# ============================================

# Number of oral disease classes
NUM_CLASSES = 6

# Input image dimensions
IMAGE_SIZE = 224

# Start with a conservative batch size because
# the GTX 1650 has 4 GB of VRAM.
BATCH_SIZE = 16

# Maximum number of training epochs.
# Early stopping may stop training before this.
NUM_EPOCHS = 15

# Number of epochs we allow without improvement
# in validation loss before stopping.
EARLY_STOPPING_PATIENCE = 3

# Learning rate for the optimizer
LEARNING_RATE = 0.0001

# Weight decay helps reduce overfitting
WEIGHT_DECAY = 0.0001


# ============================================
# DISPLAY SETTINGS
# ============================================

print("Device:", device)
print("Number of classes:", NUM_CLASSES)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Device: cuda
Number of classes: 6
Image size: 224
Batch size: 16
Maximum epochs: 15
Early stopping patience: 3
Learning rate: 0.0001
Weight decay: 0.0001


In [4]:
# ============================================
# LOAD DATA FOR PRETRAINED MODELS
# ============================================

import pandas as pd
import numpy as np

from pathlib import Path
from PIL import Image

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# ============================================
# DATASET PATHS
# ============================================

# Root directory containing the cleaned image folders
DATASET_PATH = Path(
    r"D:\AI_Diploma\Training tasks\Sprint_2\dataset"
)

# Directory containing our fixed train/validation/test CSV files
SPLIT_PATH = Path("data_splits")


# ============================================
# CLASS MAPPING
# ============================================

# Keep exactly the same class order used throughout
# the project so every model uses identical labels.
class_names = [
    "Calculus",
    "Caries",
    "Gingivitis",
    "Hypodontia",
    "Mouth Ulcer",
    "Tooth Discoloration"
]

# Convert class names into numerical labels
class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(class_names)
}

# Convert numerical labels back into class names
idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}


# ============================================
# LOAD FIXED DATA SPLITS
# ============================================

# We use the exact same splits created during EDA.
# We do NOT split the dataset again.
train_df = pd.read_csv(SPLIT_PATH / "train.csv")
val_df = pd.read_csv(SPLIT_PATH / "validation.csv")
test_df = pd.read_csv(SPLIT_PATH / "test.csv")


# ============================================
# IMAGE TRANSFORMS
# ============================================

# ImageNet statistics are required because our
# pretrained models were originally trained on ImageNet.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transformation includes realistic augmentation.
train_transform = transforms.Compose([
    # Resize every image to the model's input size
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    # Randomly flip the image horizontally
    transforms.RandomHorizontalFlip(p=0.5),

    # Simulate small differences in camera orientation
    transforms.RandomRotation(degrees=15),

    # Simulate small changes in position and distance
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.9, 1.1)
    ),

    # Simulate mild lighting differences
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),

    # Convert PIL image to PyTorch tensor
    transforms.ToTensor(),

    # Normalize using ImageNet statistics
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# Validation and test data receive NO random augmentation.
val_test_transform = transforms.Compose([
    # Resize to the model input size
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    # Convert image to tensor
    transforms.ToTensor(),

    # Apply the same ImageNet normalization
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================
# CUSTOM DATASET
# ============================================

class OralDiseaseDataset(Dataset):

    def __init__(self, dataframe, dataset_path, transform=None):
        # Reset the DataFrame index so indexing remains consistent
        self.dataframe = dataframe.reset_index(drop=True)

        # Store the root dataset directory
        self.dataset_path = Path(dataset_path)

        # Store the transformation pipeline
        self.transform = transform

    def __len__(self):
        # Return the number of images
        return len(self.dataframe)

    def __getitem__(self, index):
        # Get the row corresponding to this image
        row = self.dataframe.iloc[index]

        # Build the complete image path
        image_path = self.dataset_path / row["relative_path"]

        # Open the image and force it to RGB
        image = Image.open(image_path).convert("RGB")

        # Convert class name into its numerical label
        label = class_to_idx[row["class"]]

        # Apply preprocessing/augmentation
        if self.transform:
            image = self.transform(image)

        # Return image tensor and label
        return image, label


# ============================================
# CREATE DATASETS
# ============================================

train_dataset = OralDiseaseDataset(
    dataframe=train_df,
    dataset_path=DATASET_PATH,
    transform=train_transform
)

val_dataset = OralDiseaseDataset(
    dataframe=val_df,
    dataset_path=DATASET_PATH,
    transform=val_test_transform
)

test_dataset = OralDiseaseDataset(
    dataframe=test_df,
    dataset_path=DATASET_PATH,
    transform=val_test_transform
)


# ============================================
# CREATE DATALOADERS
# ============================================

# We start with 16 because the GTX 1650 has 4 GB VRAM.
BATCH_SIZE = 16

# 0 is safest on Windows and avoids multiprocessing issues.
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


# ============================================
# CLASS WEIGHTS
# ============================================

# These are the weights calculated earlier from
# the TRAINING set only.
class_weights = torch.tensor(
    [
        3.8132,  # Calculus
        0.6648,  # Caries
        1.3842,  # Gingivitis
        1.4086,  # Hypodontia
        0.6143,  # Mouth Ulcer
        0.8522   # Tooth Discoloration
    ],
    dtype=torch.float32
).to(device)


# ============================================
# VERIFY
# ============================================

print("Data loaded successfully.")
print("--------------------------------")
print("Training images   :", len(train_dataset))
print("Validation images :", len(val_dataset))
print("Test images       :", len(test_dataset))
print("Batch size        :", BATCH_SIZE)
print("Training batches  :", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches      :", len(test_loader))

print("\nClass weights:")
for idx, class_name in enumerate(class_names):
    print(f"{class_name:20s}: {class_weights[idx].item():.4f}")

Data loaded successfully.
--------------------------------
Training images   : 7184
Validation images : 1540
Test images       : 1540
Batch size        : 16
Training batches  : 449
Validation batches: 97
Test batches      : 97

Class weights:
Calculus            : 3.8132
Caries              : 0.6648
Gingivitis          : 1.3842
Hypodontia          : 1.4086
Mouth Ulcer         : 0.6143
Tooth Discoloration : 0.8522


In [5]:
# ============================================
# LOAD PRETRAINED RESNET18
# ============================================

# Load ResNet18 with weights pretrained on ImageNet.
# These weights contain visual features learned from
# millions of natural images.
resnet18 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Display the original final classification layer
print("Original classifier:")
print(resnet18.fc)


# ============================================
# REPLACE THE CLASSIFIER
# ============================================

# ResNet18 originally predicts 1000 ImageNet classes.
# Our dataset contains only 6 oral disease classes.
#
# resnet18.fc.in_features gives us the number of
# features produced by the ResNet before classification.
num_features = resnet18.fc.in_features

# Replace the original 1000-class classifier
# with a new classifier for our 6 disease classes.
resnet18.fc = nn.Linear(
    num_features,
    NUM_CLASSES
)


# ============================================
# MOVE MODEL TO GPU
# ============================================

# Move the complete model to our GTX 1650.
resnet18 = resnet18.to(device)


# ============================================
# DISPLAY MODEL
# ============================================

print("\nNew classifier:")
print(resnet18.fc)

print("\nModel device:")
print(next(resnet18.parameters()).device)

Original classifier:
Linear(in_features=512, out_features=1000, bias=True)

New classifier:
Linear(in_features=512, out_features=6, bias=True)

Model device:
cuda:0


In [6]:
# ============================================
# LOSS FUNCTION AND OPTIMIZER
# ============================================

# CrossEntropyLoss is appropriate for our
# six-class classification problem.
#
# The class weights make mistakes on minority
# classes contribute more to the loss.
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# AdamW is our initial optimizer.
# It works well for fine-tuning pretrained models.
optimizer = optim.AdamW(
    resnet18.parameters(),
    
    # Conservative learning rate because
    # the model already has pretrained features.
    lr=LEARNING_RATE,
    
    # Weight decay helps reduce overfitting.
    weight_decay=WEIGHT_DECAY
)


# Display the configuration
print("Loss function:")
print(criterion)

print("\nOptimizer:")
print(optimizer)

print("\nLearning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Loss function:
CrossEntropyLoss()

Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)

Learning rate: 0.0001
Weight decay: 0.0001


In [7]:
# ============================================
# EARLY STOPPING
# ============================================

class EarlyStopping:
    """
    Stop training when validation loss stops improving.

    Parameters
    ----------
    patience : int
        Number of consecutive epochs allowed without
        improvement before training stops.

    min_delta : float
        Minimum improvement required to consider the
        validation loss meaningfully better.
    """

    def __init__(self, patience=3, min_delta=0.0):
        # Number of epochs we tolerate without improvement
        self.patience = patience

        # Minimum amount of improvement required
        self.min_delta = min_delta

        # Track the best validation loss seen so far
        self.best_loss = None

        # Count consecutive epochs without improvement
        self.counter = 0

        # Indicates whether training should stop
        self.early_stop = False

    def __call__(self, val_loss):
        """
        Update early-stopping state using the current
        validation loss.
        """

        # First validation result becomes our baseline
        if self.best_loss is None:
            self.best_loss = val_loss
            return

        # Check whether validation loss improved
        if val_loss < self.best_loss - self.min_delta:

            # Store the new best validation loss
            self.best_loss = val_loss

            # Reset the non-improvement counter
            self.counter = 0

        else:

            # Validation loss did not improve
            self.counter += 1

            print(
                f"Early stopping counter: "
                f"{self.counter}/{self.patience}"
            )

            # Stop training when patience is exceeded
            if self.counter >= self.patience:
                self.early_stop = True


# ============================================
# CREATE EARLY STOPPING OBJECT
# ============================================

early_stopping = EarlyStopping(
    patience=EARLY_STOPPING_PATIENCE,
    min_delta=0.0
)

print("Early stopping initialized.")
print("Patience:", EARLY_STOPPING_PATIENCE)
print("Minimum improvement:", 0.0)

Early stopping initialized.
Patience: 3
Minimum improvement: 0.0


In [8]:
# ============================================
# TRAINING FUNCTION
# ============================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    Train the model for one complete epoch.

    Returns
    -------
    epoch_loss : float
        Average training loss for the epoch.

    epoch_accuracy : float
        Training accuracy for the epoch.
    """

    # Put the model into training mode.
    # This enables layers such as Dropout and BatchNorm
    # to behave appropriately during training.
    model.train()

    # Track total loss and correct predictions
    running_loss = 0.0
    correct = 0
    total = 0

    # Process the dataset one batch at a time
    for images, labels in loader:

        # Move images and labels to the GPU
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Clear gradients from the previous batch
        optimizer.zero_grad()

        # Forward pass:
        # images → model → class scores
        outputs = model(images)

        # Calculate weighted classification loss
        loss = criterion(outputs, labels)

        # Backpropagation:
        # calculate gradients for model parameters
        loss.backward()

        # Update the model parameters
        optimizer.step()

        # Add this batch's loss to the running total
        running_loss += loss.item() * images.size(0)

        # Get the class with the highest predicted score
        _, predicted = torch.max(outputs, 1)

        # Count correctly classified images
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate average loss across the complete epoch
    epoch_loss = running_loss / total

    # Calculate classification accuracy
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy


# ============================================
# VALIDATION FUNCTION
# ============================================

def validate(model, loader, criterion, device):
    """
    Evaluate the model on validation data.

    No gradients are calculated because the model
    is not being updated during validation.

    Returns
    -------
    epoch_loss : float
        Average validation loss.

    epoch_accuracy : float
        Validation accuracy.
    """

    # Put the model into evaluation mode.
    # This disables training-specific behavior such as
    # Dropout randomness.
    model.eval()

    # Track total loss and correct predictions
    running_loss = 0.0
    correct = 0
    total = 0

    # Disable gradient calculation to save memory
    # and speed up validation.
    with torch.no_grad():

        # Process validation data batch by batch
        for images, labels in loader:

            # Move data to the GPU
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Forward pass
            outputs = model(images)

            # Calculate validation loss
            loss = criterion(outputs, labels)

            # Accumulate loss
            running_loss += loss.item() * images.size(0)

            # Get predicted class
            _, predicted = torch.max(outputs, 1)

            # Count predictions
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Calculate average validation loss
    epoch_loss = running_loss / total

    # Calculate validation accuracy
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy


print("Training and validation functions created successfully.")

Training and validation functions created successfully.


In [14]:
# ============================================
# RESTART RESNET18 TRAINING FROM EPOCH 1
# ============================================

from pathlib import Path
import time

# ============================================
# RECREATE A FRESH RESNET18
# ============================================

# Load a completely fresh ResNet18 with ImageNet
# pretrained weights.
resnet18 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Replace the original 1000-class ImageNet classifier
# with our 6-class oral disease classifier.
num_features = resnet18.fc.in_features

resnet18.fc = nn.Linear(
    num_features,
    NUM_CLASSES
)

# Move the model to the GTX 1650.
resnet18 = resnet18.to(device)


# ============================================
# RECREATE LOSS AND OPTIMIZER
# ============================================

# Use weighted cross-entropy to compensate for
# the imbalance in our six disease classes.
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# AdamW is used for stable fine-tuning of the
# pretrained network.
optimizer = optim.AdamW(
    resnet18.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================
# TRAINING CONFIGURATION
# ============================================

# Maximum number of epochs
NUM_EPOCHS = 15

# Number of epochs allowed without validation
# loss improvement before stopping.
PATIENCE = 3

# Folder for model checkpoints
MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)

# File where the best ResNet18 model will be stored
BEST_MODEL_PATH = MODEL_DIR / "resnet18_best.pth"


# ============================================
# MIXED PRECISION
# ============================================

# Enable mixed precision when using CUDA.
# This reduces memory usage and can speed up
# training on supported NVIDIA GPUs.
USE_AMP = device.type == "cuda"

# GradScaler helps keep mixed-precision training
# numerically stable.
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


# ============================================
# TRAINING HISTORY
# ============================================

# Store metrics from every epoch so we can
# visualize learning curves later.
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}


# ============================================
# EARLY STOPPING VARIABLES
# ============================================

# Start with an infinitely large validation loss,
# so the first epoch will automatically become
# the initial best model.
best_val_loss = float("inf")

# Count consecutive epochs without improvement.
epochs_without_improvement = 0


# ============================================
# START TRAINING
# ============================================

print("=" * 70)
print("STARTING RESNET18 TRAINING FROM EPOCH 1")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):

    # Record when the epoch starts
    epoch_start = time.time()


    # ========================================
    # TRAINING PHASE
    # ========================================

    # Enable training behavior such as Dropout
    # and training-mode BatchNorm.
    resnet18.train()

    # Reset epoch statistics
    train_running_loss = 0.0
    train_correct = 0
    train_total = 0

    # Process every training batch
    for images, labels in train_loader:

        # Move images and labels to the GPU.
        images = images.to(device)
        labels = labels.to(device)

        # Remove gradients from the previous batch.
        optimizer.zero_grad(set_to_none=True)

        # Use mixed precision on CUDA.
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            # Forward pass
            outputs = resnet18(images)

            # Calculate weighted classification loss
            loss = criterion(outputs, labels)

        # Calculate gradients
        scaler.scale(loss).backward()

        # Update model parameters
        scaler.step(optimizer)

        # Update the gradient scaler
        scaler.update()

        # Accumulate total loss
        train_running_loss += loss.item() * images.size(0)

        # Convert model scores to predicted classes
        _, predicted = torch.max(outputs, 1)

        # Update accuracy counters
        train_total += labels.size(0)
        train_correct += (
            predicted == labels
        ).sum().item()

    # Calculate average training loss
    train_loss = train_running_loss / train_total

    # Calculate training accuracy
    train_accuracy = train_correct / train_total


    # ========================================
    # VALIDATION PHASE
    # ========================================

    # Switch to evaluation mode.
    resnet18.eval()

    # Reset validation statistics
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    # No gradients are required during validation.
    with torch.no_grad():

        # Process validation batches
        for images, labels in val_loader:

            # Move validation data to GPU
            images = images.to(device)
            labels = labels.to(device)

            # Use mixed precision during validation
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                # Forward pass
                outputs = resnet18(images)

                # Calculate validation loss
                loss = criterion(outputs, labels)

            # Accumulate validation loss
            val_running_loss += loss.item() * images.size(0)

            # Get predicted classes
            _, predicted = torch.max(outputs, 1)

            # Update validation counters
            val_total += labels.size(0)
            val_correct += (
                predicted == labels
            ).sum().item()

    # Calculate validation loss
    val_loss = val_running_loss / val_total

    # Calculate validation accuracy
    val_accuracy = val_correct / val_total


    # ========================================
    # SAVE HISTORY
    # ========================================

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)


    # ========================================
    # DISPLAY EPOCH RESULTS
    # ========================================

    epoch_time = time.time() - epoch_start

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )


    # ========================================
    # SAVE BEST MODEL
    # ========================================

    # A lower validation loss means the model
    # currently generalizes better.
    if val_loss < best_val_loss:

        # Update best validation loss
        best_val_loss = val_loss

        # Reset the early stopping counter
        epochs_without_improvement = 0

        # Save the model and training information
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": resnet18.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
                "class_names": class_names
            },
            BEST_MODEL_PATH
        )

        print(
            f"  -> Best model saved "
            f"(Val Loss: {val_loss:.4f})"
        )

    else:

        # Validation loss did not improve
        epochs_without_improvement += 1

        print(
            f"  -> No improvement: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )


    # ========================================
    # EARLY STOPPING
    # ========================================

    # Stop training when validation loss has
    # failed to improve for PATIENCE epochs.
    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping triggered at "
            f"epoch {epoch}."
        )

        break


# ============================================
# TRAINING COMPLETE
# ============================================

print("\n" + "=" * 70)
print("RESNET18 TRAINING COMPLETE")
print("=" * 70)

print("Best checkpoint:", BEST_MODEL_PATH)
print(f"Best validation loss: {best_val_loss:.4f}")

STARTING RESNET18 TRAINING FROM EPOCH 1
Epoch [01/15] | Train Loss: 0.5609 | Train Acc: 0.8078 | Val Loss: 0.2801 | Val Acc: 0.9039 | Time: 209.3s
  -> Best model saved (Val Loss: 0.2801)
Epoch [02/15] | Train Loss: 0.2549 | Train Acc: 0.9159 | Val Loss: 0.1060 | Val Acc: 0.9708 | Time: 199.9s
  -> Best model saved (Val Loss: 0.1060)
Epoch [03/15] | Train Loss: 0.1675 | Train Acc: 0.9456 | Val Loss: 0.0960 | Val Acc: 0.9753 | Time: 227.4s
  -> Best model saved (Val Loss: 0.0960)
Epoch [04/15] | Train Loss: 0.1134 | Train Acc: 0.9637 | Val Loss: 0.1040 | Val Acc: 0.9740 | Time: 292.8s
  -> No improvement: 1/3
Epoch [05/15] | Train Loss: 0.1173 | Train Acc: 0.9619 | Val Loss: 0.0634 | Val Acc: 0.9831 | Time: 253.3s
  -> Best model saved (Val Loss: 0.0634)
Epoch [06/15] | Train Loss: 0.1049 | Train Acc: 0.9649 | Val Loss: 0.0827 | Val Acc: 0.9740 | Time: 288.7s
  -> No improvement: 1/3
Epoch [07/15] | Train Loss: 0.0817 | Train Acc: 0.9736 | Val Loss: 0.0803 | Val Acc: 0.9812 | Time: 299.

In [15]:
# ============================================
# LOAD BEST RESNET18 CHECKPOINT
# ============================================

from pathlib import Path
import torch

# Path to the best model saved during training
BEST_MODEL_PATH = Path("saved_models/resnet18_best.pth")

# Load the saved checkpoint.
# map_location ensures the checkpoint is loaded
# correctly onto our current device.
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

# Load the best model weights
resnet18.load_state_dict(
    checkpoint["model_state_dict"]
)

# Put the model into evaluation mode
# so BatchNorm/Dropout behave correctly.
resnet18.eval()

print("Best ResNet18 checkpoint loaded successfully.")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])
print("Best validation accuracy:", checkpoint["val_accuracy"])

Best ResNet18 checkpoint loaded successfully.
Best epoch: 5
Best validation loss: 0.06342414666765503
Best validation accuracy: 0.9831168831168832


In [16]:
# ============================================
# RESNET18 - TEST SET EVALUATION
# ============================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

import numpy as np
import torch


# ============================================
# EVALUATION MODE
# ============================================

# Make sure the model is in evaluation mode.
# This disables training-specific behavior such
# as Dropout and uses evaluation behavior for BatchNorm.
resnet18.eval()


# ============================================
# STORE PREDICTIONS AND TRUE LABELS
# ============================================

all_predictions = []
all_labels = []


# Disable gradients because we are only evaluating
# the trained model.
with torch.no_grad():

    # Process the test set batch by batch
    for images, labels in test_loader:

        # Move images and labels to the GPU
        images = images.to(device)
        labels = labels.to(device)

        # Generate predictions
        outputs = resnet18(images)

        # Select the class with the highest score
        _, predictions = torch.max(outputs, 1)

        # Move results back to CPU and store them
        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# Convert results into NumPy arrays
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)


# ============================================
# CALCULATE OVERALL METRICS
# ============================================

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

test_macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

test_weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)


# ============================================
# DISPLAY OVERALL RESULTS
# ============================================

print("=" * 60)
print("RESNET18 - TEST RESULTS")
print("=" * 60)

print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Macro F1 : {test_macro_f1:.4f}")
print(f"Test Weighted F1: {test_weighted_f1:.4f}")


# ============================================
# CLASSIFICATION REPORT
# ============================================

print("\nClassification Report:")
print("----------------------")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================
# CONFUSION MATRIX
# ============================================

conf_matrix = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")
print("-----------------")
print(conf_matrix)

RESNET18 - TEST RESULTS
Test Accuracy : 0.9805
Test Macro F1 : 0.9762
Test Weighted F1: 0.9805

Classification Report:
----------------------
                     precision    recall  f1-score   support

           Calculus     0.9552    0.9552    0.9552        67
             Caries     0.9576    0.9948    0.9759       386
         Gingivitis     0.9731    0.9731    0.9731       186
         Hypodontia     0.9836    0.9836    0.9836       183
        Mouth Ulcer     1.0000    0.9928    0.9964       417
Tooth Discoloration     0.9931    0.9535    0.9729       301

           accuracy                         0.9805      1540
          macro avg     0.9771    0.9755    0.9762      1540
       weighted avg     0.9809    0.9805    0.9805      1540

Confusion Matrix:
-----------------
[[ 64   0   3   0   0   0]
 [  0 384   2   0   0   0]
 [  3   0 181   2   0   0]
 [  0   2   0 180   0   1]
 [  0   2   0   0 414   1]
 [  0  13   0   1   0 287]]


In [17]:
# ============================================
# LOAD PRETRAINED EFFICIENTNET-B0
# ============================================

# Load EfficientNet-B0 with weights pretrained
# on ImageNet.
efficientnet_b0 = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

# Display the original classifier so we can see
# how many classes ImageNet expects.
print("Original classifier:")
print(efficientnet_b0.classifier)


# ============================================
# REPLACE THE CLASSIFIER
# ============================================

# EfficientNet-B0's classifier is a Sequential
# module. The final Linear layer produces the
# ImageNet predictions.
#
# We get the number of input features going into
# that final layer.
num_features = (
    efficientnet_b0.classifier[-1].in_features
)

# Replace the ImageNet classifier with a new
# classifier for our six oral disease classes.
efficientnet_b0.classifier[-1] = nn.Linear(
    num_features,
    NUM_CLASSES
)


# ============================================
# MOVE MODEL TO GPU
# ============================================

# Move the model to our NVIDIA GTX 1650.
efficientnet_b0 = efficientnet_b0.to(device)


# ============================================
# DISPLAY NEW CLASSIFIER
# ============================================

print("\nNew classifier:")
print(efficientnet_b0.classifier[-1])

print("\nModel device:")
print(next(efficientnet_b0.parameters()).device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\gigabite/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:05<00:00, 3.69MB/s]


Original classifier:
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

New classifier:
Linear(in_features=1280, out_features=6, bias=True)

Model device:
cuda:0


In [18]:
# ============================================
# EFFICIENTNET-B0 LOSS AND OPTIMIZER
# ============================================

# Use the same class-weighted CrossEntropyLoss
# we used for ResNet18 so the comparison remains fair.
efficientnet_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Create a new AdamW optimizer specifically for
# EfficientNet-B0's parameters.
efficientnet_optimizer = optim.AdamW(
    efficientnet_b0.parameters(),

    # Use the same learning rate as ResNet18
    # for a controlled comparison.
    lr=LEARNING_RATE,

    # Use the same weight decay as ResNet18
    # to keep regularization consistent.
    weight_decay=WEIGHT_DECAY
)


# ============================================
# DISPLAY CONFIGURATION
# ============================================

print("Loss function:")
print(efficientnet_criterion)

print("\nOptimizer:")
print(efficientnet_optimizer)

print("\nLearning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Loss function:
CrossEntropyLoss()

Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)

Learning rate: 0.0001
Weight decay: 0.0001


In [19]:
# ============================================
# EFFICIENTNET-B0 TRAINING WITH PROGRESS BAR
# ============================================

import time
from pathlib import Path
from tqdm.auto import tqdm


# ============================================
# TRAINING CONFIGURATION
# ============================================

# Maximum number of epochs
NUM_EPOCHS = 15

# Stop training after this many epochs without
# validation loss improvement
PATIENCE = 3

# Create directory for saved model checkpoints
MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)

# Path for the best EfficientNet-B0 checkpoint
BEST_EFFICIENTNET_PATH = (
    MODEL_DIR / "efficientnet_b0_best.pth"
)


# ============================================
# MIXED PRECISION
# ============================================

# Enable mixed precision when using the GPU.
USE_AMP = device.type == "cuda"

# GradScaler improves numerical stability when
# using mixed precision during backpropagation.
efficientnet_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


# ============================================
# TRAINING HISTORY
# ============================================

# Store metrics from every epoch so we can
# visualize training after it finishes.
efficientnet_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}


# ============================================
# EARLY STOPPING VARIABLES
# ============================================

# Start with an infinitely large validation loss.
best_val_loss = float("inf")

# Count consecutive epochs without improvement.
epochs_without_improvement = 0


# ============================================
# START TRAINING
# ============================================

print("=" * 70)
print("STARTING EFFICIENTNET-B0 TRAINING")
print("=" * 70)


for epoch in range(1, NUM_EPOCHS + 1):

    # Record the start time of the epoch
    epoch_start = time.time()


    # ========================================
    # TRAINING PHASE
    # ========================================

    # Put the model into training mode.
    efficientnet_b0.train()

    # Reset epoch statistics.
    train_running_loss = 0.0
    train_correct = 0
    train_total = 0


    # Create a green progress bar for the batches.
    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{NUM_EPOCHS}",
        unit="batch",
        colour="green"
    )


    # Process every training batch
    for batch_idx, (images, labels) in enumerate(progress_bar):

        # Move images and labels to the GPU
        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients from the previous batch
        efficientnet_optimizer.zero_grad(
            set_to_none=True
        )

        # Mixed-precision forward pass
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            # Generate predictions
            outputs = efficientnet_b0(images)

            # Calculate class-weighted loss
            loss = efficientnet_criterion(
                outputs,
                labels
            )

        # Calculate gradients
        efficientnet_scaler.scale(loss).backward()

        # Update model parameters
        efficientnet_scaler.step(
            efficientnet_optimizer
        )

        # Update gradient scaler
        efficientnet_scaler.update()

        # Accumulate loss
        train_running_loss += (
            loss.item() * images.size(0)
        )

        # Get predicted classes
        _, predicted = torch.max(outputs, 1)

        # Update counters
        train_total += labels.size(0)
        train_correct += (
            predicted == labels
        ).sum().item()


        # Calculate current running accuracy
        current_accuracy = train_correct / train_total

        # Update the progress bar with useful
        # information while training.
        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{current_accuracy:.4f}"
        )


    # Calculate epoch-level training metrics
    train_loss = train_running_loss / train_total
    train_accuracy = train_correct / train_total


    # ========================================
    # VALIDATION PHASE
    # ========================================

    # Put the model into evaluation mode.
    efficientnet_b0.eval()

    # Reset validation statistics
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0


    # Disable gradients during validation
    with torch.no_grad():

        # Validation progress bar
        validation_bar = tqdm(
            val_loader,
            desc="Validation",
            unit="batch",
            colour="green",
            leave=False
        )

        for images, labels in validation_bar:

            # Move validation data to GPU
            images = images.to(device)
            labels = labels.to(device)

            # Mixed-precision forward pass
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                # Generate predictions
                outputs = efficientnet_b0(images)

                # Calculate validation loss
                loss = efficientnet_criterion(
                    outputs,
                    labels
                )

            # Accumulate validation loss
            val_running_loss += (
                loss.item() * images.size(0)
            )

            # Get predicted classes
            _, predicted = torch.max(outputs, 1)

            # Update counters
            val_total += labels.size(0)
            val_correct += (
                predicted == labels
            ).sum().item()


    # Calculate validation metrics
    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total


    # ========================================
    # SAVE HISTORY
    # ========================================

    efficientnet_history[
        "train_loss"
    ].append(train_loss)

    efficientnet_history[
        "train_accuracy"
    ].append(train_accuracy)

    efficientnet_history[
        "val_loss"
    ].append(val_loss)

    efficientnet_history[
        "val_accuracy"
    ].append(val_accuracy)


    # ========================================
    # EPOCH SUMMARY
    # ========================================

    epoch_time = time.time() - epoch_start

    print(
        f"\nEpoch {epoch}/{NUM_EPOCHS} Summary"
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f}"
    )

    print(
        f"Val Loss:   {val_loss:.4f} | "
        f"Val Acc:    {val_accuracy:.4f}"
    )

    print(
        f"Epoch Time: {epoch_time:.1f}s"
    )


    # ========================================
    # SAVE BEST MODEL
    # ========================================

    # Check whether validation loss improved.
    if val_loss < best_val_loss:

        # Update best validation loss
        best_val_loss = val_loss

        # Reset early-stopping counter
        epochs_without_improvement = 0

        # Save the best checkpoint
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    efficientnet_b0.state_dict(),
                "optimizer_state_dict":
                    efficientnet_optimizer.state_dict(),
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
                "class_names": class_names
            },
            BEST_EFFICIENTNET_PATH
        )

        print(
            f"✅ Best model saved | "
            f"Val Loss: {val_loss:.4f}"
        )

    else:

        # Validation loss did not improve
        epochs_without_improvement += 1

        print(
            f"⚠️ No improvement: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )


    # ========================================
    # EARLY STOPPING
    # ========================================

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\n🛑 Early stopping triggered at "
            f"epoch {epoch}."
        )

        break


# ============================================
# TRAINING COMPLETE
# ============================================

print("\n" + "=" * 70)
print("EFFICIENTNET-B0 TRAINING COMPLETE")
print("=" * 70)

print(
    "Best checkpoint:",
    BEST_EFFICIENTNET_PATH
)

print(
    f"Best validation loss: "
    f"{best_val_loss:.4f}"
)

STARTING EFFICIENTNET-B0 TRAINING


Epoch 1/15:   0%|          | 0/449 [00:00<?, ?batch/s]

Validation:   0%|          | 0/97 [00:00<?, ?batch/s]


Epoch 1/15 Summary
Train Loss: 0.7417 | Train Acc: 0.7622
Val Loss:   0.2152 | Val Acc:    0.9299
Epoch Time: 147.4s
✅ Best model saved | Val Loss: 0.2152


Epoch 2/15:   0%|          | 0/449 [00:00<?, ?batch/s]

...

Validation:   0%|          | 0/97 [00:00<?, ?batch/s]


Epoch 10/15 Summary
Train Loss: 0.0347 | Train Acc: 0.9884
Val Loss:   0.0361 | Val Acc:    0.9929
Epoch Time: 134.7s
⚠️ No improvement: 2/3


Epoch 11/15:   0%|          | 0/449 [00:00<?, ?batch/s]

Validation:   0%|          | 0/97 [00:00<?, ?batch/s]


Epoch 11/15 Summary
Train Loss: 0.0335 | Train Acc: 0.9884
Val Loss:   0.0558 | Val Acc:    0.9896
Epoch Time: 134.8s
⚠️ No improvement: 3/3

🛑 Early stopping triggered at epoch 11.

EFFICIENTNET-B0 TRAINING COMPLETE
Best checkpoint: saved_models\efficientnet_b0_best.pth
Best validation loss: 0.0099


In [20]:
# ============================================
# LOAD BEST EFFICIENTNET-B0 CHECKPOINT
# ============================================

# Path to the checkpoint with the lowest validation loss
BEST_EFFICIENTNET_PATH = (
    "saved_models/efficientnet_b0_best.pth"
)

# Load the checkpoint onto our current device
checkpoint = torch.load(
    BEST_EFFICIENTNET_PATH,
    map_location=device
)

# Load the saved model weights
efficientnet_b0.load_state_dict(
    checkpoint["model_state_dict"]
)

# Put the model into evaluation mode
# so BatchNorm and Dropout behave correctly.
efficientnet_b0.eval()

print("Best EfficientNet-B0 checkpoint loaded.")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])
print("Best validation accuracy:", checkpoint["val_accuracy"])

Best EfficientNet-B0 checkpoint loaded.
Best epoch: 8
Best validation loss: 0.009860891208102527
Best validation accuracy: 0.9980519480519481


In [21]:
# ============================================
# EFFICIENTNET-B0 - TEST SET EVALUATION
# ============================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)


# ============================================
# EVALUATION MODE
# ============================================

# Put the model into evaluation mode.
# This disables training-specific behavior such
# as Dropout and changes BatchNorm to evaluation mode.
efficientnet_b0.eval()


# ============================================
# STORE PREDICTIONS AND TRUE LABELS
# ============================================

# Lists used to collect predictions from all
# test batches.
all_predictions = []
all_labels = []


# Disable gradient calculation because the model
# is only being evaluated.
with torch.no_grad():

    # Process the test dataset batch by batch
    for images, labels in test_loader:

        # Move the test batch to the GPU
        images = images.to(device)
        labels = labels.to(device)

        # Generate predictions
        outputs = efficientnet_b0(images)

        # Select the class with the highest score
        _, predictions = torch.max(outputs, 1)

        # Move predictions and labels back to CPU
        # so they can be processed by scikit-learn.
        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# Convert the lists to NumPy arrays
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)


# ============================================
# CALCULATE OVERALL METRICS
# ============================================

# Overall classification accuracy
test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

# Macro F1 gives every disease class equal importance.
test_macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

# Weighted F1 accounts for the number of samples
# in each class.
test_weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)


# ============================================
# DISPLAY OVERALL RESULTS
# ============================================

print("=" * 60)
print("EFFICIENTNET-B0 - TEST RESULTS")
print("=" * 60)

print(
    f"Test Accuracy    : {test_accuracy:.4f}"
)

print(
    f"Test Macro F1    : {test_macro_f1:.4f}"
)

print(
    f"Test Weighted F1 : {test_weighted_f1:.4f}"
)


# ============================================
# CLASSIFICATION REPORT
# ============================================

print("\nClassification Report:")
print("----------------------")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================
# CONFUSION MATRIX
# ============================================

# Calculate the confusion matrix
conf_matrix_efficientnet = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print("-----------------")

# Convert the matrix to a labeled DataFrame
# so rows and columns are easy to interpret.
confusion_df_efficientnet = pd.DataFrame(
    conf_matrix_efficientnet,
    index=class_names,
    columns=class_names
)

print(confusion_df_efficientnet)

EFFICIENTNET-B0 - TEST RESULTS
Test Accuracy    : 0.9961
Test Macro F1    : 0.9948
Test Weighted F1 : 0.9961

Classification Report:
----------------------
                     precision    recall  f1-score   support

           Calculus     0.9853    1.0000    0.9926        67
             Caries     1.0000    0.9896    0.9948       386
         Gingivitis     0.9840    0.9892    0.9866       186
         Hypodontia     0.9892    1.0000    0.9946       183
        Mouth Ulcer     1.0000    1.0000    1.0000       417
Tooth Discoloration     1.0000    1.0000    1.0000       301

           accuracy                         0.9961      1540
          macro avg     0.9931    0.9965    0.9948      1540
       weighted avg     0.9961    0.9961    0.9961      1540


Confusion Matrix:
-----------------
                     Calculus  Caries  Gingivitis  Hypodontia  Mouth Ulcer  \
Calculus                   67       0           0           0            0   
Caries                      0     382 

In [22]:
# ============================================
# FULL EFFICIENTNET-B0 CONFUSION MATRIX
# ============================================

import pandas as pd

# Convert the previously calculated confusion matrix
# into a labeled DataFrame.
efficientnet_confusion_df = pd.DataFrame(
    conf_matrix_efficientnet,
    index=class_names,
    columns=class_names
)

# Display the complete confusion matrix.
# Rows = actual classes
# Columns = predicted classes
print("EfficientNet-B0 Confusion Matrix")
print("=" * 60)

print(efficientnet_confusion_df)

# Optional: display the total number of mistakes.
total_errors = (
    conf_matrix_efficientnet.sum()
    - conf_matrix_efficientnet.trace()
)

print("\nTotal test errors:", total_errors)

EfficientNet-B0 Confusion Matrix
                     Calculus  Caries  Gingivitis  Hypodontia  Mouth Ulcer  \
Calculus                   67       0           0           0            0   
Caries                      0     382           3           1            0   
Gingivitis                  1       0         184           1            0   
Hypodontia                  0       0           0         183            0   
Mouth Ulcer                 0       0           0           0          417   
Tooth Discoloration         0       0           0           0            0   

                     Tooth Discoloration  
Calculus                               0  
Caries                                 0  
Gingivitis                             0  
Hypodontia                             0  
Mouth Ulcer                            0  
Tooth Discoloration                  301  

Total test errors: 6


In [23]:
# ============================================
# LOAD PRETRAINED MOBILENETV3-SMALL
# ============================================

# Load MobileNetV3-Small with ImageNet-pretrained
# weights.
mobilenet_v3 = models.mobilenet_v3_small(
    weights=models.MobileNet_V3_Small_Weights.DEFAULT
)

# Display the original classifier.
# MobileNetV3 uses a Sequential classifier.
print("Original classifier:")
print(mobilenet_v3.classifier)


# ============================================
# REPLACE THE CLASSIFIER
# ============================================

# Get the number of features entering the final
# classification layer.
num_features = (
    mobilenet_v3.classifier[-1].in_features
)

# Replace the original ImageNet classifier
# with a classifier for our six oral disease classes.
mobilenet_v3.classifier[-1] = nn.Linear(
    num_features,
    NUM_CLASSES
)


# ============================================
# MOVE MODEL TO GPU
# ============================================

# Move the model to the NVIDIA GTX 1650.
mobilenet_v3 = mobilenet_v3.to(device)


# ============================================
# DISPLAY NEW CLASSIFIER
# ============================================

print("\nNew classifier:")
print(mobilenet_v3.classifier[-1])

print("\nModel device:")
print(next(mobilenet_v3.parameters()).device)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to C:\Users\gigabite/.cache\torch\hub\checkpoints\mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:02<00:00, 3.71MB/s]

Original classifier:
Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=1000, bias=True)
)

New classifier:
Linear(in_features=1024, out_features=6, bias=True)

Model device:
cuda:0


In [24]:
# ============================================
# MOBILENETV3-SMALL LOSS AND OPTIMIZER
# ============================================

# Use the same class-weighted CrossEntropyLoss
# used for ResNet18 and EfficientNet-B0.
mobilenet_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Create a separate AdamW optimizer because
# this optimizer must update MobileNet's parameters.
mobilenet_optimizer = optim.AdamW(
    mobilenet_v3.parameters(),

    # Keep the same learning rate for a fair comparison
    # between all pretrained models.
    lr=LEARNING_RATE,

    # Keep the same weight decay as the other models.
    weight_decay=WEIGHT_DECAY
)


# ============================================
# DISPLAY CONFIGURATION
# ============================================

print("Loss function:")
print(mobilenet_criterion)

print("\nOptimizer:")
print(mobilenet_optimizer)

print("\nLearning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Loss function:
CrossEntropyLoss()

Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)

Learning rate: 0.0001
Weight decay: 0.0001


In [25]:
# ============================================
# TRAIN MOBILENETV3-SMALL
# ============================================

import time
from pathlib import Path
from tqdm.auto import tqdm


# ============================================
# TRAINING CONFIGURATION
# ============================================

# Maximum number of epochs
NUM_EPOCHS = 15

# Stop training after this many epochs without
# improvement in validation loss
PATIENCE = 3

# Create the directory for model checkpoints
MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)

# Path for the best MobileNetV3-Small model
BEST_MOBILENET_PATH = (
    MODEL_DIR / "mobilenet_v3_small_best.pth"
)


# ============================================
# MIXED PRECISION
# ============================================

# Enable mixed precision when CUDA is available.
USE_AMP = device.type == "cuda"

# GradScaler helps maintain numerical stability
# during mixed-precision training.
mobilenet_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


# ============================================
# TRAINING HISTORY
# ============================================

# Store metrics from every epoch so we can later
# compare MobileNet with the other models.
mobilenet_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}


# ============================================
# EARLY STOPPING VARIABLES
# ============================================

# Start with infinitely large validation loss.
best_val_loss = float("inf")

# Count consecutive epochs without improvement.
epochs_without_improvement = 0


# ============================================
# START TRAINING
# ============================================

print("=" * 70)
print("STARTING MOBILENETV3-SMALL TRAINING")
print("=" * 70)


for epoch in range(1, NUM_EPOCHS + 1):

    # Record the start time of this epoch
    epoch_start = time.time()


    # ========================================
    # TRAINING PHASE
    # ========================================

    # Put MobileNet into training mode.
    mobilenet_v3.train()

    # Reset training statistics
    train_running_loss = 0.0
    train_correct = 0
    train_total = 0


    # Create a green progress bar for training batches.
    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{NUM_EPOCHS}",
        unit="batch",
        colour="green"
    )


    # Process all training batches
    for images, labels in progress_bar:

        # Move images and labels to the GTX 1650
        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients from the previous batch
        mobilenet_optimizer.zero_grad(
            set_to_none=True
        )

        # Use mixed precision on CUDA
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            # Forward pass
            outputs = mobilenet_v3(images)

            # Calculate class-weighted loss
            loss = mobilenet_criterion(
                outputs,
                labels
            )

        # Calculate gradients
        mobilenet_scaler.scale(loss).backward()

        # Update model weights
        mobilenet_scaler.step(
            mobilenet_optimizer
        )

        # Update gradient scaler
        mobilenet_scaler.update()

        # Accumulate training loss
        train_running_loss += (
            loss.item() * images.size(0)
        )

        # Get predicted classes
        _, predicted = torch.max(outputs, 1)

        # Update training counters
        train_total += labels.size(0)
        train_correct += (
            predicted == labels
        ).sum().item()

        # Calculate running accuracy
        current_accuracy = (
            train_correct / train_total
        )

        # Display live information on the progress bar
        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{current_accuracy:.4f}"
        )


    # Calculate final training metrics
    train_loss = train_running_loss / train_total
    train_accuracy = train_correct / train_total


    # ========================================
    # VALIDATION PHASE
    # ========================================

    # Switch MobileNet to evaluation mode.
    mobilenet_v3.eval()

    # Reset validation statistics
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0


    # No gradients are required for validation.
    with torch.no_grad():

        # Create a green validation progress bar.
        validation_bar = tqdm(
            val_loader,
            desc="Validation",
            unit="batch",
            colour="green",
            leave=False
        )

        for images, labels in validation_bar:

            # Move validation data to GPU
            images = images.to(device)
            labels = labels.to(device)

            # Use mixed precision during validation
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                # Forward pass
                outputs = mobilenet_v3(images)

                # Calculate validation loss
                loss = mobilenet_criterion(
                    outputs,
                    labels
                )

            # Accumulate validation loss
            val_running_loss += (
                loss.item() * images.size(0)
            )

            # Get predictions
            _, predicted = torch.max(outputs, 1)

            # Update validation counters
            val_total += labels.size(0)
            val_correct += (
                predicted == labels
            ).sum().item()


    # Calculate validation metrics
    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total


    # ========================================
    # SAVE TRAINING HISTORY
    # ========================================

    mobilenet_history[
        "train_loss"
    ].append(train_loss)

    mobilenet_history[
        "train_accuracy"
    ].append(train_accuracy)

    mobilenet_history[
        "val_loss"
    ].append(val_loss)

    mobilenet_history[
        "val_accuracy"
    ].append(val_accuracy)


    # ========================================
    # EPOCH SUMMARY
    # ========================================

    epoch_time = time.time() - epoch_start

    print(
        f"\nEpoch {epoch}/{NUM_EPOCHS} Summary"
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f}"
    )

    print(
        f"Val Loss:   {val_loss:.4f} | "
        f"Val Acc:    {val_accuracy:.4f}"
    )

    print(
        f"Epoch Time: {epoch_time:.1f}s"
    )


    # ========================================
    # SAVE BEST MODEL
    # ========================================

    # Save the model whenever validation loss
    # reaches a new minimum.
    if val_loss < best_val_loss:

        # Update best validation loss
        best_val_loss = val_loss

        # Reset the early stopping counter
        epochs_without_improvement = 0

        # Save the best model checkpoint
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    mobilenet_v3.state_dict(),
                "optimizer_state_dict":
                    mobilenet_optimizer.state_dict(),
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
                "class_names": class_names
            },
            BEST_MOBILENET_PATH
        )

        print(
            f"✅ Best model saved | "
            f"Val Loss: {val_loss:.4f}"
        )

    else:

        # Validation loss did not improve
        epochs_without_improvement += 1

        print(
            f"⚠️ No improvement: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )


    # ========================================
    # EARLY STOPPING
    # ========================================

    # Stop when validation loss has failed to
    # improve for PATIENCE consecutive epochs.
    if epochs_without_improvement >= PATIENCE:

        print(
            f"\n🛑 Early stopping triggered at "
            f"epoch {epoch}."
        )

        break


# ============================================
# TRAINING COMPLETE
# ============================================

print("\n" + "=" * 70)
print("MOBILENETV3-SMALL TRAINING COMPLETE")
print("=" * 70)

print(
    "Best checkpoint:",
    BEST_MOBILENET_PATH
)

print(
    f"Best validation loss: "
    f"{best_val_loss:.4f}"
)

STARTING MOBILENETV3-SMALL TRAINING


Epoch 1/15:   0%|          | 0/449 [00:00<?, ?batch/s]

Validation:   0%|          | 0/97 [00:00<?, ?batch/s]


Epoch 1/15 Summary
Train Loss: 0.8541 | Train Acc: 0.7024
Val Loss:   0.3357 | Val Acc:    0.8864
Epoch Time: 156.4s
✅ Best model saved | Val Loss: 0.3357


Epoch 2/15:   0%|          | 0/449 [00:00<?, ?batch/s]

...

Validation:   0%|          | 0/97 [00:00<?, ?batch/s]


Epoch 14/15 Summary
Train Loss: 0.0498 | Train Acc: 0.9847
Val Loss:   0.0435 | Val Acc:    0.9896
Epoch Time: 51.3s
⚠️ No improvement: 2/3


Epoch 15/15:   0%|          | 0/449 [00:00<?, ?batch/s]

Validation:   0%|          | 0/97 [00:00<?, ?batch/s]


Epoch 15/15 Summary
Train Loss: 0.0323 | Train Acc: 0.9889
Val Loss:   0.0351 | Val Acc:    0.9896
Epoch Time: 55.4s
⚠️ No improvement: 3/3

🛑 Early stopping triggered at epoch 15.

MOBILENETV3-SMALL TRAINING COMPLETE
Best checkpoint: saved_models\mobilenet_v3_small_best.pth
Best validation loss: 0.0322


In [26]:
# ============================================
# LOAD BEST MOBILENETV3-SMALL CHECKPOINT
# ============================================

# Path to the best MobileNet checkpoint
BEST_MOBILENET_PATH = (
    "saved_models/mobilenet_v3_small_best.pth"
)

# Load the checkpoint onto our current device
checkpoint = torch.load(
    BEST_MOBILENET_PATH,
    map_location=device
)

# Load the saved model weights
mobilenet_v3.load_state_dict(
    checkpoint["model_state_dict"]
)

# Put the model into evaluation mode
# so Dropout and BatchNorm behave correctly.
mobilenet_v3.eval()

print("Best MobileNetV3-Small checkpoint loaded.")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])
print("Best validation accuracy:", checkpoint["val_accuracy"])

Best MobileNetV3-Small checkpoint loaded.
Best epoch: 12
Best validation loss: 0.032212390548469465
Best validation accuracy: 0.9883116883116884


In [27]:
# ============================================
# MOBILENETV3-SMALL - TEST SET EVALUATION
# ============================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# Make sure the model is in evaluation mode
mobilenet_v3.eval()

# Store all predictions and true labels
all_predictions = []
all_labels = []

# Disable gradients because we are only evaluating
# the already-trained model.
with torch.no_grad():

    # Process the test set batch by batch
    for images, labels in test_loader:

        # Move data to the GPU
        images = images.to(device)
        labels = labels.to(device)

        # Generate predictions
        outputs = mobilenet_v3(images)

        # Select the class with the highest score
        _, predictions = torch.max(outputs, 1)

        # Move results to CPU for scikit-learn
        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# Convert lists to NumPy arrays
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Calculate overall accuracy
test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

# Calculate macro F1.
# Every disease class receives equal importance.
test_macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

# Calculate weighted F1.
test_weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

# Display overall results
print("=" * 60)
print("MOBILENETV3-SMALL - TEST RESULTS")
print("=" * 60)

print(f"Test Accuracy    : {test_accuracy:.4f}")
print(f"Test Macro F1    : {test_macro_f1:.4f}")
print(f"Test Weighted F1 : {test_weighted_f1:.4f}")

# Display per-class precision, recall and F1
print("\nClassification Report:")
print("----------------------")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# Calculate and display the confusion matrix
mobilenet_conf_matrix = confusion_matrix(
    all_labels,
    all_predictions
)

mobilenet_confusion_df = pd.DataFrame(
    mobilenet_conf_matrix,
    index=class_names,
    columns=class_names
)

print("\nConfusion Matrix:")
print("-----------------")
print(mobilenet_confusion_df)

MOBILENETV3-SMALL - TEST RESULTS
Test Accuracy    : 0.9786
Test Macro F1    : 0.9691
Test Weighted F1 : 0.9787

Classification Report:
----------------------
                     precision    recall  f1-score   support

           Calculus     0.8667    0.9701    0.9155        67
             Caries     0.9843    0.9715    0.9778       386
         Gingivitis     0.9730    0.9677    0.9704       186
         Hypodontia     0.9630    0.9945    0.9785       183
        Mouth Ulcer     0.9952    0.9904    0.9928       417
Tooth Discoloration     0.9898    0.9701    0.9799       301

           accuracy                         0.9786      1540
          macro avg     0.9620    0.9774    0.9691      1540
       weighted avg     0.9793    0.9786    0.9787      1540


Confusion Matrix:
-----------------
                     Calculus  Caries  Gingivitis  Hypodontia  Mouth Ulcer  \
Calculus                   65       0           2           0            0   
Caries                      3     37